<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/58/Uber_logo_2018.svg/1024px-Uber_logo_2018.svg.png" alt="UBER LOGO" width="50%" />

# UBER Pickups 

# import libraries

In [ ]:
import pandas as pd
import plotly.express as px

# Read data

In [ ]:
zone = pd.read_csv('data/taxi-zone-lookup.csv')
print('Lines / Columns : ', zone.shape)

In [ ]:
files = ["data/uber-trip-data/uber-raw-data-apr14.csv", "data/uber-trip-data/uber-raw-data-may14.csv", "data/uber-trip-data/uber-raw-data-jun14.csv", 'data/uber-trip-data/uber-raw-data-jul14.csv', 'data/uber-trip-data/uber-raw-data-aug14.csv', 'data/uber-trip-data/uber-raw-data-sep14.csv']

raw_2014 = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print('Lines / Columns : ', raw_2014.shape)

In [ ]:
raw_2015 = pd.read_csv('data/uber-raw-data-janjune-15.csv')
print('Lines / Columns : ', raw_2015.shape)

# Explore dataset

In [ ]:
dataframes = [zone, raw_2014, raw_2015]
[print(f"Display of dataset: {display(df.head())}") for df in dataframes]

We observe :
- date time MM/DD/YYYY
- lowercase and uppercase
- sorted raw data 



In [ ]:
# Basic stats
dataframes = {'zone':zone, 'raw_2014':raw_2014, 'raw_2015':raw_2015}

for name, df in dataframes.items():
    print()
    print('-------------------Dataframe : ', name, '-------------------')
    print(f"Rows :  \t {df.shape[0]}")
    print(f"Columns : \t {df.shape[1]}")

    print()
    print("Basics statistics: ")
    df_desc = df.describe(include='all') # numeric variables and catégorielles
    display(df_desc)

    print("Percentage of missing values")
    df_missing_values = df.isnull().sum()/df.shape[0]*100
    print(f"{name} : \n{df_missing_values}")
print('Done...')

We observe missing values on raw_2015

In [ ]:
# Lowercase dataframes
for name, df in dataframes.items():
    df.columns = df.columns.str.lower() # lowercase columns names
    for col in df.select_dtypes(include=["object", 'str']).columns: # avoid warning for multi execution
        df[col] = df[col].str.lower()
        print()
    print('-------------------',name,'-------------------')
    print(df.head(1))
    for col in df.select_dtypes(include=["object", 'str']).columns: # avoid warning for multi execution
        df[col] = df[col].str.lower()
        print()
    print('-------------------',name,'-------------------')
    print(df.head(1))

In [ ]:
# raw_2014['base'].isin(raw_2015['dispatching_base_num']).value_counts()

On retrouve le nombre de lignes exact de raw_2014. Aucune base ne diffère des dispatching_base_num.

Pour s'en assurer, comparons les valeurs uniques.

In [ ]:
b14 = set(raw_2014['base'].unique())
b15_dispatch = set(raw_2015['dispatching_base_num'].unique())
b15_licence = set(raw_2015['affiliated_base_num'].unique())
print(len(b14), len(b15_dispatch), len(b15_licence), len(b14 & b15_dispatch & b15_licence))

# Treat datetime

In [ ]:
# import pandas as pd

# raw_2014["date/time"], raw_2015["pickup_date"] = [
#     pd.to_datetime(c, format=f) for c, f in [(raw_2014["date/time"], "%m/%d/%Y %H:%M:%S"),
#                                             (raw_2015["pickup_date"], "%Y-%m-%d %H:%M:%S")]
# ]

In [ ]:
cols = [raw_2014["date/time"], raw_2015["pickup_date"]]

converted = [pd.to_datetime(c, format=f) for c, f in zip(cols, ("%m/%d/%Y %H:%M:%S", "%Y-%m-%d %H:%M:%S")) # zip to use c and f together
]

raw_2014["date/time"], raw_2015["pickup_date"] = converted
print(raw_2014["date/time"].dtypes) # datetime64[ns]
print(raw_2015["pickup_date"].dtypes)
print([c.dtype for c in converted])  # # M8[ns]

# Fusionner raw_2015 et zone

In [ ]:
raw_2015 = raw_2015.merge(zone, on="locationid", how="left")

In [ ]:
raw_2015.head(1)

In [ ]:
raw_2015.to_csv("data/2015.csv", index=False)

# Treat missing values

Une position gps imputée fausserait les clusters. On supprime les lignes.

In [ ]:
before = raw_2015.shape[0]
raw_2015 = raw_2015.dropna(subset='affiliated_base_num') # drop lines where values are missing. 
missing_values = raw_2015.isnull().sum()/raw_2015.shape[0]*100
after = raw_2015.shape[0]
deleted_rows_pct = (before - after) / before * 100
print('deleted_rows_pct : ', deleted_rows_pct)
print(missing_values)

# Analyze one hour on one day

In [ ]:
raw_2015.head(1)

In [ ]:
raw_2014['date/time'].dtype

In [ ]:
raw_2015['pickup_date'].dtype

In [ ]:
begin = pd.Timestamp('2014-04-01 00:00:00')
end = begin + pd.Timedelta(hours=1)

t1 = raw_2014[(raw_2014['date/time'] >= begin) & (raw_2014['date/time'] < end)]

In [ ]:
t1

In [ ]:
bases_median = t1.groupby('base')[['lat', 'lon']].median().reset_index()

In [ ]:
print(bases_median)

In [ ]:
fig = px.scatter_map(t1, lat="lat", lon="lon", color="base", zoom=10, map_style="carto-positron")
fig.show()

Les bases b02617 et b02764 se superposent

# Create bases

In [ ]:
bases = raw_2014.groupby('base')[['lat', 'lon']].median().reset_index()

In [ ]:
raw_2014[['base', 'lon', 'lat']].to_csv("data/bases.csv", index=False)

In [ ]:
# print(bases)

# Search for duplicated values

In [ ]:
duplicated_values = bases[bases.duplicated(subset=["lat", "lon"], keep=False)]
print(duplicated_values.sort_values(['base', "lat", "lon"]))

In [ ]:
with pd.option_context("display.precision", 15):
    print(bases_median)

Les bases ont pratiquement les mêmes valeurs.

In [ ]:
# bases_t1 = bases_median.round({"lat": 4, "lon": 4})
# print(bases_t1[bases_t1.duplicated(subset=["lat", "lon"], keep=False)])

# Fusionner raw_2015 et bases

In [ ]:
raw_2015 = raw_2015.merge(bases[["base", "lon", "lat"]], left_on="dispatching_base_num", right_on="base", how="left").drop(columns="base")
print(raw_2015[["dispatching_base_num", "lon", "lat"]])
print(raw_2015[["lon", "lat"]])

In [ ]:
raw_2015.head(1)

# Detect outlyers

# Create sample

In [ ]:
mask = raw_2015[raw_2015['dispatching_base_num'].isin(bases_median['base'])]
n = 5000  # taille cible, choix provisoire
frac = n / len(mask)
sample = mask.groupby('dispatching_base_num').sample(frac=frac, random_state=42)
print(frac, len(sample))

In [ ]:
sample.to_csv("data/sample.csv", index=False)

Question : how use locationid